In [0]:
catalog_name = dbutils.widgets.get("catalog_name")

# # Load staging data with customer changes
# staging_data = [
#     ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
#     ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
#     ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
#     ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
# ]

# df_staging_customers = spark.createDataFrame(
#     staging_data,
#     ["customer_id", "full_name", "email", "city", "segment", "account_type"]
# )

file_path = "/Volumes/ws_databricks/default/sharedfiles/Customer Update.csv"

# load customer update data from csv
df_staging_customers = spark.read.format("csv") \
                            .option("header", "true") \
                            .load(file_path)

df_staging_customers.createOrReplaceTempView("staging_customers")
print("Staging data loaded. Temp view 'df_staging_customers' is available.")
display(df_staging_customers)

In [0]:
%sql
    
-- SCD Type 2: Close changed records
MERGE INTO IDENTIFIER(:catalog_name || '.2_silver.dim_customer') AS target
USING staging_customers AS source
  ON target.customer_id = source.customer_id
  AND target.is_current = true
WHEN MATCHED AND (
    target.city         <> source.city
    OR target.segment   <> source.segment
    OR target.email     <> source.email
    OR target.account_type <> source.account_type
)
THEN UPDATE SET
  target.valid_to    = current_timestamp(),
  target.is_current  = false;

-- SCD Type 2: Insert new versions and new customers
INSERT INTO IDENTIFIER(:catalog_name || '.2_silver.dim_customer')
  (customer_id, full_name, email, city, segment, account_type, valid_from, valid_to, is_current)
SELECT
  s.customer_id,
  s.full_name,
  s.email,
  s.city,
  s.segment,
  s.account_type,
  current_timestamp()           AS valid_from,
  to_timestamp('9999-12-31')    AS valid_to,
  true                          AS is_current
FROM staging_customers s
LEFT ANTI JOIN IDENTIFIER(:catalog_name || '.2_silver.dim_customer') t
  ON s.customer_id = t.customer_id
  AND t.is_current = true;